# Terra Signal — Análise exploratória
**Grupo 2 · Hackathon Databricks / ASN.Rocks**

> **A pergunta da empresa:** *"Como podemos usar estes dados, e as tendências que eles mostram, para melhorar a nossa situação de churn e ganhar mais dinheiro?"*

A Terra Signal está perdendo clientes e não sabe onde agir. Este notebook conta essa investigação como uma **história em 5 atos**: cada ato faz uma pergunta, os dados respondem, e a resposta levanta a pergunta seguinte.

| Ato | A pergunta | O papel na história |
|---|---|---|
| 1 | Quem são os clientes da Terra Signal? | **O cenário:** com o que estamos lidando |
| 2 | **Quando** os clientes saem? | **A primeira pista:** o problema tem hora para acontecer |
| 3 | **Quem** sai? | **O suspeito:** um perfil concentra as saídas |
| 4 | **Onde está o dinheiro** que está sendo perdido? | **O que está em jogo:** quem sai vale mais |
| 5 | **Por que** saem? | **O motivo:** o que os próprios clientes dizem |
| — | Conclusão | **O desfecho:** do diagnóstico às alavancas |

**Como ler cada passo:** o título é a pergunta. Logo abaixo, eu digo quais dados entram, como calculo e por quê. Depois do código vêm **o que os dados mostram**, a **ressalva** (quando há) e, no fim de cada ato, **o gancho** para a próxima pergunta.

## Convenções gerais

- **Base:** só o `history.csv` (5.634 clientes × 23 colunas). Cada linha é um cliente, num retrato único do momento em que a base foi montada. O `inference.csv` **não** entra na análise exploratória.
- **Referência:** a taxa de cada grupo é sempre comparada com a média da base (26,7%).
- **Cores dos gráficos:** vermelho para quem sai ou para o destaque; cinza para o contexto.
- As siglas *R1*, *R3* etc. apontam para as respostas da organização às nossas perguntas (`respostas_grupo2.md`).

In [0]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# Vermelho = quem sai ou o destaque. Cinza = contexto.
VERMELHO, CINZA, CINZA_ESCURO = "#d03b3b", "#b9b8b0", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})

def pct(v, casas=1):
    # 0.267 -> '26,7%'
    return f"{v:.{casas}%}".replace(".", ",")

def brl(v, casas=2):
    # 1234.5 -> 'R$ 1.234,50'
    s = f"{abs(v):,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return f"-R$ {s}" if v < 0 else f"R$ {s}"

---
# Ato 1 — Quem são os clientes da Terra Signal?

> **O cenário.** Antes de procurar culpados, eu preciso conhecer o lugar. Quantos clientes são, o que sabemos de cada um e quantos já foram embora?

**Objetivo do ato:** descrever a base antes de olhar para o churn, para saber com o que estamos lidando.

### 1.1 · Qual é o tamanho da base?

- **Como:** `pd.read_csv("history.csv")`.

In [0]:
history = pd.read_csv("./history.csv")
print(f"{history.shape[0]} clientes × {history.shape[1]} colunas")
history.head(3).T

**O que os dados mostram:** 5.634 clientes × 23 colunas.

### 1.2 · O que precisa ser corrigido antes das contas?

- **Dados:** base inteira.
- **Como e por quê:** corrijo só o que impede as contas, sem ainda fazer a limpeza completa.

| Problema | Linhas | Causa (resposta da organização) | O que eu faço | Por quê |
|---|---|---|---|---|
| `tenure = "unknown"` | 19 | Dado perdido na origem (*R3*) | `pd.to_numeric(..., errors="coerce")` → vira vazio (`NaN`) | Sem isso a coluna é lida como texto. As 19 linhas ficam **fora** das contas que usam `tenure`. |
| `TotalCharges` vazio | 278 | 10 com `tenure = 0` (ainda sem fatura, *R10*) e 268 com dado perdido (*R11*) | `pd.to_numeric(..., errors="coerce")` → vazio | Precisa ser número para calcular estatísticas. |
| `PhoneService` com `yes`/`no` | 70 | Erro de padronização (*R13*) | `.str.capitalize()` → `Yes`/`No` | Senão `yes` e `Yes` virariam categorias diferentes. |

Também crio a **variável de churn** em número, para facilitar as contas: 1 = saiu, 0 = ficou. Com ela, **a média de `churn` em qualquer grupo é a taxa de churn desse grupo**:

> taxa de churn do grupo = Σ churn ÷ nº de clientes do grupo = clientes que saíram ÷ clientes no grupo

In [0]:
df = history.copy()

# Conto os problemas ANTES de corrigir, para registrar
antes = pd.Series({
    "tenure = 'unknown'": (df["tenure"] == "unknown").sum(),
    "TotalCharges vazio": pd.to_numeric(df["TotalCharges"], errors="coerce").isna().sum(),
    "PhoneService em minúsculas": df["PhoneService"].isin(["yes", "no"]).sum(),
}, name="linhas")

df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")              # 'unknown' vira NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")  # vazio vira NaN
df["PhoneService"] = df["PhoneService"].str.capitalize()                 # yes/no -> Yes/No

df["churn"] = (df["Churn"] == "Yes").astype(int)                         # 1 = saiu, 0 = ficou
TAXA_MEDIA = df["churn"].mean()

display(antes.to_frame())

# Os tamanhos de base que aparecem ao longo do notebook
tem_tenure = df["tenure"].notna()
pd.Series({
    "Base inteira": len(df),
    "Com tenure conhecido": tem_tenure.sum(),
    "Saíram": df["churn"].sum(),
    "Saíram, com tenure conhecido": (tem_tenure & (df["churn"] == 1)).sum(),
    "Ficaram, com tenure conhecido": (tem_tenure & (df["churn"] == 0)).sum(),
    "Com internet": (df["InternetService"] != "No").sum(),
}, name="clientes").to_frame()

**Ressalva:** a imputação (estimar `tenure` por `TotalCharges ÷ MonthlyCharges`, por exemplo) **não** é feita aqui. Ela fica para a etapa de limpeza e modelagem. As 19 linhas sem `tenure` são 0,3% da base.

### 1.3 · Quantos clientes saíram?

- **Dados:** base inteira (5.634).
- **Como:** `value_counts()` e a proporção de cada valor.

In [0]:
saidas = df["Churn"].value_counts().to_frame("clientes")
saidas["%"] = (saidas["clientes"] / len(df)).map(pct)
saidas

**O que os dados mostram:** 4.130 ficaram (73,3%) e 1.504 saíram (**26,7%**). Cerca de 1 em cada 4.

### 1.4 · Como são os números da base?

- **Dados:** `tenure` (5.615 com valor), `MonthlyCharges` (5.634), `TotalCharges` (5.356) e `MonthlyIncome` (5.634).
- **Como:** `describe()`, que ignora os vazios automaticamente. Mostra contagem, média, desvio-padrão, mínimo, quartis e máximo.

In [0]:
NUMERICAS = ["tenure", "MonthlyCharges", "TotalCharges", "MonthlyIncome"]
df[NUMERICAS].describe().round(1)

**O que os dados mostram:**
- `tenure` vai de 0 a 72 meses; mediana de 29; um quarto da base tem até 9 meses.
- `MonthlyCharges` vai de R\$ 18 a R\$ 119.
- `TotalCharges` acompanha `tenure`, porque é basicamente a soma das mensalidades (*R12*).

**Ressalva:** `MonthlyIncome` vem de uma base externa sem garantia de precisão (*R15*); a distribuição é quase uniforme e não muda com o churn.

### 1.5 · Quais perfis predominam?

- **Dados:** base inteira, 16 variáveis categóricas.
- **Como:** para cada variável, `value_counts(normalize=True)`, mostrando a categoria mais frequente e a distribuição completa.

In [0]:
CATEGORICAS = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines",
               "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
               "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"]

linhas = []
for col in CATEGORICAS:
    dist = df[col].value_counts(normalize=True)
    linhas.append({
        "variável": col,
        "categoria mais frequente": f"{dist.index[0]} ({pct(dist.iloc[0], 0)})",
        "distribuição completa": " · ".join(f"{k}: {pct(v, 0)}" for k, v in dist.items()),
    })
pd.DataFrame(linhas).set_index("variável")

**O que os dados mostram:** predominam o **contrato mensal (55%)** e a **internet de fibra (44%)**. As duas voltam no Ato 3.

**O que o Ato 1 revela:** a base tem 5.634 clientes, e 26,7% saíram. É uma carteira dominada por contrato mensal e fibra.

**➜ O gancho:** um em cada quatro clientes foi embora. A primeira coisa que eu quero saber é **quando** isso acontece: logo no começo, ou depois de anos?

---
# Ato 2 — Quando os clientes saem?

> **A primeira pista.** Se os clientes saíssem em qualquer momento, o churn seria um problema espalhado, difícil de atacar. Se saem numa fase específica, existe uma janela para agir.

**Objetivo do ato:** entender em que fase da vida do cliente as saídas acontecem.

**Filtro de todo o ato:** só clientes com `tenure` conhecido (5.615). Os 19 `unknown` ficam fora, porque não dá para saber em que fase eles estão.

### 2.1 · Quem saiu tinha menos tempo de casa?

- **Dados:** saíram com `tenure` conhecido (1.496) e ficaram com `tenure` conhecido (4.119).
- **Como:** `groupby("Churn")["tenure"].describe()`. O quantil `p` de um grupo é o valor que deixa uma fração `p` dos clientes do grupo abaixo dele. O boxplot mostra os mesmos quartis graficamente.
- **Por quê:** se a saída fosse igual em qualquer fase, quem saiu e quem ficou teriam tempos de casa parecidos.

In [0]:
com_tenure = df[df["tenure"].notna()].copy()

quartis = com_tenure.groupby("Churn")["tenure"].describe()[["count", "25%", "50%", "75%"]]
quartis.columns = ["clientes", "Q1 (25%)", "mediana", "Q3 (75%)"]
quartis = quartis.rename(index={"Yes": "Saiu", "No": "Ficou"}).astype(int)
display(quartis)

fig, ax = plt.subplots(figsize=(8, 2.6))
grupos = [com_tenure.loc[com_tenure["churn"] == 0, "tenure"], com_tenure.loc[com_tenure["churn"] == 1, "tenure"]]
with warnings.catch_warnings():          # 'vert' funciona em qualquer versão do matplotlib; só silencio o aviso das mais novas
    warnings.simplefilter("ignore")
    caixas = ax.boxplot(grupos, vert=False, widths=0.55, patch_artist=True, medianprops=dict(color="white", linewidth=2))
for caixa, cor in zip(caixas["boxes"], [CINZA, VERMELHO]):
    caixa.set(facecolor=cor, edgecolor=cor)
ax.set_yticks([1, 2], ["Ficou", "Saiu"])
ax.set(title=f"Quem saiu ficou bem menos tempo: metade saiu até o {quartis.loc['Saiu', 'mediana']}º mês",
       xlabel="meses de casa (tenure)")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**O que os dados mostram:**

| | Q1 (25%) | Mediana | Q3 (75%) |
|---|---|---|---|
| Saiu | 2 meses | 10 meses | 30 meses |
| Ficou | 14 meses | 37 meses | 61 meses |

O Q1 de quem ficou (14) é maior que a mediana de quem saiu (10): os grupos quase não se sobrepõem no início. **Quem sai, sai cedo.**

**Ressalva:** essa comparação olha **só para quem saiu** ("quando foram embora?"). O passo seguinte faz a pergunta inversa, que é a útil para o negócio.

### 2.2 · Como separar os clientes por fase da vida?

- **Dados:** 5.615 clientes.
- **Como:** `pd.qcut(tenure, q=4, retbins=True)`. Os cortes são os quantis da **base inteira**. O cliente vai para o quartil `k` quando `corte(k-1) < tenure ≤ corte(k)` (o primeiro inclui o zero). Os rótulos são **gerados a partir dos cortes**, e não digitados.
- **Por quê:** quartis dão grupos de tamanho parecido, então nenhuma taxa fica baseada em poucos clientes. Gerar os rótulos a partir dos cortes evita que fiquem errados se os dados mudarem.

In [0]:
_, cortes = pd.qcut(com_tenure["tenure"], q=4, retbins=True)
cortes = cortes.astype(int)

# Rótulos gerados a partir dos cortes: 'Q1: 0–9 meses', 'Q2: 10–29 meses', ...
rotulos = [f"Q{k + 1}: {cortes[k] + (1 if k > 0 else 0)}–{cortes[k + 1]} meses" for k in range(4)]
com_tenure["quartil_tenure"] = pd.qcut(com_tenure["tenure"], q=4, labels=rotulos)
df["quartil_tenure"] = com_tenure["quartil_tenure"]      # fica vazio para os 19 sem tenure

print("Cortes:", cortes.tolist())
com_tenure["quartil_tenure"].value_counts().sort_index().to_frame("clientes")

**O que os dados mostram:** cortes em 0 · 9 · 29 · 55 · 72 meses, com 1.480 · 1.361 · 1.379 · 1.395 clientes.

**Ressalva:** os grupos não têm exatamente 25% cada porque `tenure` é inteiro e muitos clientes têm o mesmo valor. Todos os clientes com o mesmo `tenure` vão para o mesmo grupo.

### 2.3 · Em que fase o risco é maior?

- **Dados:** os 4 quartis do passo anterior (quem saiu e quem ficou juntos).
- **Como:** `groupby("quartil_tenure")["churn"].agg(...)`. A taxa do quartil é: clientes do quartil que saíram ÷ clientes do quartil.
- **Gráfico:** barras com a linha tracejada da média; a barra de **maior taxa** fica em vermelho (calculado, não fixo); eixo começando em zero para não exagerar as diferenças.

In [0]:
por_quartil = com_tenure.groupby("quartil_tenure", observed=True)["churn"].agg(
    clientes="size", saíram="sum", taxa_churn="mean")
display(por_quartil.assign(taxa_churn=por_quartil["taxa_churn"].map(pct)))
print(f"Maior taxa ÷ média da base = {por_quartil['taxa_churn'].max() / TAXA_MEDIA:.2f}".replace(".", ","))

cores = [VERMELHO if t == por_quartil["taxa_churn"].max() else CINZA for t in por_quartil["taxa_churn"]]
fig, ax = plt.subplots(figsize=(8, 3.4))
barras = ax.bar(por_quartil.index.astype(str), por_quartil["taxa_churn"], color=cores, width=0.6)
for b, t, n in zip(barras, por_quartil["taxa_churn"], por_quartil["clientes"]):
    ax.text(b.get_x() + b.get_width() / 2, t + 0.01, f"{pct(t)}\n(n={n})", ha="center", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", pad=1), zorder=3)
ax.axhline(TAXA_MEDIA, color=CINZA_ESCURO, ls="--", lw=1, zorder=1)
ax.text(3.45, TAXA_MEDIA + 0.01, f"média: {pct(TAXA_MEDIA)}", ha="right", color=CINZA_ESCURO, fontsize=9)
ax.set(title=f"Quanto menos tempo de casa, mais churn: {pct(por_quartil['taxa_churn'].max(), 0)} no primeiro quartil",
       ylabel="taxa de churn", ylim=(0, 0.62))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**O que os dados mostram:**

| Quartil | Clientes | Saíram | Taxa |
|---|---|---|---|
| Q1: 0–9 meses | 1.480 | 726 | **49,1%** |
| Q2: 10–29 meses | 1.361 | 387 | 28,4% |
| Q3: 30–55 meses | 1.379 | 272 | 19,7% |
| Q4: 56–72 meses | 1.395 | 111 | 8,0% |

"Quase o dobro da média" vem de 49,1% ÷ 26,7% ≈ 1,84.

**Ressalva:** a base é uma foto, sem datas. Para quem saiu, o `tenure` parou no mês da saída; para quem ficou, continua contando. Então 49% **não** significa "49% dos clientes novos cancelam em 9 meses": é a proporção de clientes dessa faixa que aparecem como saída na foto. A comparação entre as faixas continua valendo.

**O que o Ato 2 revela:** o churn é um problema de **começo de relacionamento**. Um quarto dos clientes que saem vai embora até o 2º mês e metade até o 10º; entre os clientes com até 9 meses de casa, 49% saíram, contra 8% entre os que têm 56 meses ou mais. As ações de retenção devem acontecer cedo.

**➜ O gancho:** já sei **quando**. Mas nem todo cliente novo sai: metade deles fica. O que diferencia os que vão embora? É hora de perguntar **quem** sai.

---
# Ato 3 — Quem sai?

> **O suspeito.** Com a pista do tempo na mão, eu procuro um perfil. Existe um tipo de cliente que concentra as saídas, ou elas estão espalhadas por toda a base?

**Objetivo do ato:** descobrir se existe um perfil que concentra as saídas.

**Filtro:** base inteira (5.634), exceto onde indicado. Contrato, internet, pagamento e serviços não têm valores faltantes.

### 3.1 · O churn muda com o contrato, a internet e o pagamento?

- **Dados:** 5.634 clientes.
- **Como:** uma função `taxa_por(coluna)` faz o `groupby` + taxa de churn; o gráfico pinta de vermelho as categorias **acima da média** (26,7%) e mostra o `n` de cada barra.
- **Por quê destas três:** são as variáveis com as maiores diferenças de churn entre categorias.

In [0]:
def taxa_por(coluna, dados=None):
    # Taxa de churn de cada categoria de uma coluna
    dados = df if dados is None else dados
    t = dados.groupby(coluna, observed=True)["churn"].agg(clientes="size", saíram="sum", taxa_churn="mean")
    return t.sort_values("taxa_churn", ascending=False)

def grafico_taxa(tabela, titulo, ax):
    # Barras horizontais; vermelho para as categorias acima da média da base
    t = tabela.sort_values("taxa_churn")
    cores = [VERMELHO if v > TAXA_MEDIA else CINZA for v in t["taxa_churn"]]
    barras = ax.barh(t.index.astype(str), t["taxa_churn"], color=cores, height=0.6)
    for b, v, n in zip(barras, t["taxa_churn"], t["clientes"]):
        ax.text(v + 0.01, b.get_y() + b.get_height() / 2, f"{pct(v, 0)} (n={n})", va="center", fontsize=8.5,
                bbox=dict(facecolor="white", edgecolor="none", pad=1), zorder=3)
    ax.axvline(TAXA_MEDIA, color=CINZA_ESCURO, ls="--", lw=1, zorder=1)
    ax.set(title=titulo, xlim=(0, 0.75))
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
    ax.grid(axis="y", visible=False)

TRES_FORTES = ["Contract", "InternetService", "PaymentMethod"]
for col in TRES_FORTES:
    display(taxa_por(col).assign(taxa_churn=lambda d: d["taxa_churn"].map(lambda v: pct(v, 0))))

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, col in zip(axes, TRES_FORTES):
    grafico_taxa(taxa_por(col), col, ax)
fig.suptitle(f"Contrato mensal, fibra e cheque eletrônico ficam bem acima da média (linha tracejada, {pct(TAXA_MEDIA)})", y=1.03)
plt.tight_layout(); plt.show()

t_contrato = taxa_por("Contract")["taxa_churn"]
print(f"Mensal ÷ 2 anos = {t_contrato['Month-to-month'] / t_contrato['Two year']:.0f} vezes")

**O que os dados mostram:**

| Variável | Categoria (clientes) → taxa de churn |
|---|---|
| Contrato | Mensal (3.099) **43%** · 1 ano (1.188) 12% · 2 anos (1.347) **3%** |
| Internet | Fibra (2.477) **42%** · DSL (1.924) 19% · Sem internet (1.233) 7% |
| Pagamento | Cheque eletrônico (1.888) **45%** · Cheque por correio (1.283) 19% · Transferência (1.229) 17% · Cartão (1.234) 16% |

O mensal sai cerca de 13 vezes mais que o de 2 anos.

**Ressalva:**
- O churn inclui cancelamento por inadimplência (*R1*). Parte do churn do cheque eletrônico (que não é automático) pode ser **falta de pagamento**, e não insatisfação.
- Quem paga com cheque eletrônico também tem mais contrato mensal e mais fibra. Comparando só clientes parecidos (mesmo contrato, internet e faixa de tempo de casa), a diferença do cheque eletrônico cai de 28 para 9 pontos (notebook `01_eda_terra_signal.ipynb`, seção 2.3).

### 3.2 · Os serviços adicionais protegem?

- **Filtro:** só clientes **com internet** (4.401). Quem não tem internet não pode contratar adicionais (*R14*); incluí-los misturaria "não quis" com "não podia".
- **Como:** conto quantos dos 4 adicionais o cliente contratou (`n_adicionais`). Depois, calculo a taxa de churn por adicional (contratou × não contratou) e por `n_adicionais`.
- **Por quê destes 4:** são serviços de proteção e suporte. Streaming ficou de fora porque a diferença de churn ali é pequena (~33% × ~30%).

In [0]:
ADICIONAIS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]

com_internet = df[df["InternetService"] != "No"].copy()
com_internet["n_adicionais"] = (com_internet[ADICIONAIS] == "Yes").sum(axis=1)
print(f"Clientes com internet: {len(com_internet)}")

# Taxa de churn por adicional: quem não contratou x quem contratou
por_adicional = pd.DataFrame({
    col: taxa_por(col, com_internet)["taxa_churn"].rename({"No": "sem o adicional", "Yes": "com o adicional"})
    for col in ADICIONAIS}).T[["sem o adicional", "com o adicional"]]
display(por_adicional.map(lambda v: pct(v, 0)))

# Taxa de churn pela quantidade de adicionais
por_n = taxa_por("n_adicionais", com_internet).sort_index()
display(por_n.assign(taxa_churn=por_n["taxa_churn"].map(lambda v: pct(v, 0))).T)

cores = [VERMELHO if v > TAXA_MEDIA else CINZA for v in por_n["taxa_churn"]]
fig, ax = plt.subplots(figsize=(7, 3.2))
barras = ax.bar(por_n.index.astype(str), por_n["taxa_churn"], color=cores, width=0.6)
for b, v, n in zip(barras, por_n["taxa_churn"], por_n["clientes"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{pct(v, 0)}\n(n={n})", ha="center", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", pad=1), zorder=3)
ax.axhline(TAXA_MEDIA, color=CINZA_ESCURO, ls="--", lw=1, zorder=1)
ax.set(title="Quanto mais adicionais, menos churn (só clientes com internet)", ylabel="taxa de churn",
       xlabel="nº de adicionais contratados (0 a 4)", ylim=(0, 0.7))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**O que os dados mostram:**

| Nº de adicionais | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| Clientes | 1.005 | 1.189 | 1.094 | 745 | 368 |
| Taxa | **56%** | 40% | 24% | 13% | **6%** |

Segurança online e suporte técnico são os mais fortes: ~42% de churn sem, ~15% com.

**Ressalva:** é **associação, não causa**. Clientes mais engajados contratam mais e também ficam mais. Comparando só clientes parecidos, a diferença entre ter 0 ou 1 adicional e ter 2 ou mais cai de 30 para 6 pontos; segurança online (11 pontos) e suporte técnico (9) se sustentam, backup e proteção de dispositivo não (notebook `01_eda_terra_signal.ipynb`, seção 2.4).

### 3.3 · O perfil do cliente muda o churn?

- **Dados:** 5.634 clientes.
- **Como:** taxa de churn por `SeniorCitizen`, `Partner`, `Dependents`, `PaperlessBilling` e `gender`.
- **Por quê está aqui:** para mostrar que esses efeitos existem, mas são moderados perto de contrato, internet e pagamento.

In [0]:
DEMOGRAFICAS = ["SeniorCitizen", "Partner", "Dependents", "PaperlessBilling", "gender"]

linhas = []
for col in DEMOGRAFICAS:
    t = taxa_por(col)
    linhas.append({
        "variável": col,
        "maior taxa": f"{t.index[0]}: {pct(t['taxa_churn'].iloc[0], 0)} (n={t['clientes'].iloc[0]})",
        "menor taxa": f"{t.index[-1]}: {pct(t['taxa_churn'].iloc[-1], 0)} (n={t['clientes'].iloc[-1]})",
        "diferença (p.p.)": round((t["taxa_churn"].iloc[0] - t["taxa_churn"].iloc[-1]) * 100),
    })
pd.DataFrame(linhas).set_index("variável").sort_values("diferença (p.p.)", ascending=False)

**O que os dados mostram:** idosos 42% (× 24%), sem cônjuge 33% (× 20%), sem dependentes 32% (× 15%), fatura digital 34% (× 17%). Gênero: 27% × 26%, sem diferença.

**Ressalva:** parte desses efeitos é perfil. Entre clientes parecidos, o efeito de ter cônjuge some (de 13 pontos para zero) e o de dependentes cai de 16 para 3; idoso (8) e fatura digital (7) se sustentam em parte.

### 3.4 · O risco do começo vale para todos os contratos?

- **Dados:** 5.615 clientes (precisa de `tenure`).
- **Como:** `pivot_table(index="Contract", columns="quartil_tenure", values="churn", aggfunc="mean")`, e o mesmo com `aggfunc="size"` para mostrar o `n` em cada célula.
- **Por quê:** junta o Ato 2 (quando) com o Ato 3 (quem).

In [0]:
ORDEM_CONTRATO = ["Month-to-month", "One year", "Two year"]

taxa_cel = com_tenure.pivot_table(index="Contract", columns="quartil_tenure", values="churn",
                                  aggfunc="mean", observed=True).reindex(ORDEM_CONTRATO)
n_cel = com_tenure.pivot_table(index="Contract", columns="quartil_tenure", values="churn",
                               aggfunc="size", observed=True).reindex(ORDEM_CONTRATO)

fig, ax = plt.subplots(figsize=(8.5, 3.2))
ax.imshow(taxa_cel.values, cmap="Reds", vmin=0, vmax=0.6, aspect="auto")
for i in range(taxa_cel.shape[0]):
    for j in range(taxa_cel.shape[1]):
        t, n = taxa_cel.iat[i, j], n_cel.iat[i, j]
        ax.text(j, i, f"{pct(t, 0)}\nn={n}", ha="center", va="center", fontsize=9,
                color="white" if t > 0.35 else "#0b0b0b")
ax.set_xticks(range(taxa_cel.shape[1]), taxa_cel.columns.astype(str))
ax.set_yticks(range(taxa_cel.shape[0]), ["Mensal", "1 ano", "2 anos"])
ax.set_title("O risco do começo está no contrato mensal (taxa de churn por contrato e tempo de casa)"); ax.grid(False)
for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

**O que os dados mostram:**

| Contrato | Q1 (0–9) | Q2 (10–29) | Q3 (30–55) | Q4 (56–72) |
|---|---|---|---|---|
| Mensal | **52%** (n=1.378) | 38% (958) | 34% (588) | 20% (166) |
| 1 ano | 11% (62) | 8% (284) | 13% (493) | 12% (345) |
| 2 anos | 0% (40) | 1% (119) | 3% (298) | 4% (884) |

O risco do começo do relacionamento está quase todo no **contrato mensal**.

**Ressalva:** "1 ano / Q1" (62) e "2 anos / Q1" (40) têm poucos clientes, então essas taxas são pouco confiáveis. Por isso o `n` aparece no gráfico.

### 3.5 · Onde está o volume de saídas?

- **Dados:** 5.634 clientes.
- **Como:** `groupby(["Contract", "InternetService", "PaymentMethod"])`, que gera até 36 segmentos. Para cada segmento: taxa = saíram ÷ clientes; % da base = clientes ÷ 5.634; % das saídas = saíram ÷ 1.504. A tabela é ordenada pelo **número de saídas** e mostra os 8 primeiros.
- **Por quê destas três variáveis:** são as três mais fortes do Passo 3.1.
- **Por quê ordenar por saídas, e não por taxa:** um segmento pequeno pode ter taxa alta e pouco impacto. O que importa para a estratégia é onde está o **volume** de saídas.

In [0]:
def resumo_segmento(chaves):
    s = df.groupby(chaves, observed=True)["churn"].agg(clientes="size", saíram="sum", taxa_churn="mean")
    s["% da base"] = s["clientes"] / len(df)
    s["% das saídas"] = s["saíram"] / df["churn"].sum()
    return s.sort_values("saíram", ascending=False)

def formatar(s):
    return s.assign(**{c: s[c].map(lambda v: pct(v, 0)) for c in ["taxa_churn", "% da base", "% das saídas"]})

print("Os 8 segmentos com mais saídas (contrato + internet + pagamento)")
display(formatar(resumo_segmento(TRES_FORTES).head(8)))

print("Só contrato + internet")
display(formatar(resumo_segmento(["Contract", "InternetService"]).head(3)))

**O que os dados mostram:**

| Segmento | Clientes | % da base | Taxa | % das saídas |
|---|---|---|---|---|
| Mensal + fibra | 1.698 | 30% | 55% | **62%** |
| Mensal + fibra + cheque eletrônico | 1.038 | 18% | 60% | **41%** |

**O que o Ato 3 revela:** o churn se concentra em **contrato mensal + fibra**, sobretudo com cheque eletrônico e sem adicionais. Menos de um terço da base responde por quase dois terços das saídas. Alavancas candidatas: migração para contrato com fidelidade, débito automático e oferta de segurança/suporte, todas no começo do relacionamento.

**➜ O gancho:** o perfil que mais sai é justamente o da fibra, o serviço mais caro. Então a pergunta deixa de ser só "quantos clientes?" e passa a ser **"quanto dinheiro?"**.

---
# Ato 4 — Onde está o dinheiro que está sendo perdido?

> **O que está em jogo.** A Terra Signal foi clara: o objetivo é ganhar mais dinheiro, e alguns clientes valem mais que outros. Contar cabeças não basta.

**Objetivo do ato:** medir quanto de receita sai junto com os clientes, porque o objetivo financeiro da empresa é ganhar mais dinheiro (*R21*).

**Medida:** `MonthlyCharges`, o valor do último mês faturado (*R8*). Para quem saiu, é o último valor pago antes de cancelar. **Por quê esta e não `TotalCharges`:** `MonthlyCharges` mede quanto o cliente vale **por mês**, independentemente de quanto tempo ficou, e não tem valores faltantes.

**Filtro:** base inteira (5.634).

### 4.1 · Quem sai pesa mais no dinheiro do que na contagem?

- **Como:** somo as mensalidades de todos e de quem saiu, e comparo duas proporções: a fatia de **clientes** e a fatia de **dinheiro**. Se todos pagassem o mesmo valor, as duas seriam iguais.

In [0]:
peso = df.groupby("Churn").agg(clientes=("churn", "size"), soma_mensalidades=("MonthlyCharges", "sum"),
                               mensalidade_média=("MonthlyCharges", "mean"))
peso["% dos clientes"] = peso["clientes"] / peso["clientes"].sum()
peso["% do dinheiro"] = peso["soma_mensalidades"] / peso["soma_mensalidades"].sum()
peso = peso.rename(index={"Yes": "Saíram", "No": "Ficaram"}).loc[["Saíram", "Ficaram"]]

display(pd.DataFrame({
    "clientes": peso["clientes"],
    "% dos clientes": peso["% dos clientes"].map(pct),
    "soma das mensalidades": peso["soma_mensalidades"].map(brl),
    "% do dinheiro": peso["% do dinheiro"].map(pct),
    "mensalidade média": peso["mensalidade_média"].map(brl),
}))

media_saiu, media_ficou = peso.loc["Saíram", "mensalidade_média"], peso.loc["Ficaram", "mensalidade_média"]
print(f"Quem saiu pagava, em média, {brl(media_saiu - media_ficou)} a mais ({pct(media_saiu / media_ficou - 1, 0)} a mais).")

**O que os dados mostram:** quem saiu é 26,7% dos clientes, mas 30,8% da soma das mensalidades. Quem sai **paga mais**: R\$ 74,63 contra R\$ 60,96, ou seja, R\$ 13,67 (22%) a mais. A explicação é a fibra: os clientes de fibra pagam as mensalidades mais altas e também são os que mais saem (Ato 3).

**Por que isso importa:** contar só **clientes** perdidos subestima o problema. A empresa está perdendo justamente os clientes que **pagam mais**, então reter um cliente que sai vale, em média, mais que reter um cliente típico da base.

**Ressalva:** os R\$ 112 mil **não** são uma perda por mês. Os clientes saíram em momentos diferentes ao longo dos ~7 anos; o valor é a soma das últimas mensalidades de todos eles. Por isso eu **não** multiplico esse total por 12.

### 4.2 · Quanto vale evitar parte das saídas?

- **Premissa:** a empresa não definiu o valor de um cliente (*R23*). Adoto: **valor de um cliente retido = 12 × `MonthlyCharges`**.
- **Por quê 12 meses:** é um horizonte curto e conservador; não há multa por cancelamento nem histórico de reajuste (*R12*). Fica registrado como premissa para ser refinado na modelagem.
- **Filtro:** clientes que saíram do segmento **mensal + fibra**. **Por quê este segmento:** foi o identificado no Ato 3 (30% da base, 62% das saídas).
- **Como:** clientes retidos = saídas do segmento × % evitada; receita bruta preservada = soma das mensalidades dessas saídas × 12 × % evitada.

In [0]:
MESES_PREMISSA = 12      # premissa: um cliente retido vale 12 mensalidades

saidas_alvo = df[(df["Contract"] == "Month-to-month") & (df["InternetService"] == "Fiber optic") & (df["churn"] == 1)]
print(f"Saídas do segmento mensal + fibra: {len(saidas_alvo)} | mensalidade média: {brl(saidas_alvo['MonthlyCharges'].mean())}")

cenarios = pd.DataFrame({"% das saídas evitadas": [0.10, 0.20, 0.30]})
cenarios["clientes retidos"] = (len(saidas_alvo) * cenarios["% das saídas evitadas"]).round().astype(int)
cenarios["receita bruta preservada (12 meses)"] = (
    saidas_alvo["MonthlyCharges"].sum() * MESES_PREMISSA * cenarios["% das saídas evitadas"])

cenarios.assign(**{
    "% das saídas evitadas": cenarios["% das saídas evitadas"].map(lambda v: pct(v, 0)),
    "receita bruta preservada (12 meses)": cenarios["receita bruta preservada (12 meses)"].map(lambda v: brl(v, 0)),
}).set_index("% das saídas evitadas")

**O que os dados mostram:** são 928 saídas no segmento, com mensalidade média de R\$ 86,74.

| % das saídas evitadas | Clientes retidos | Receita bruta preservada (12 meses) |
|---|---|---|
| 10% | 93 | R\$ 96.591 |
| 20% | 186 | R\$ 193.182 |
| 30% | 278 | R\$ 289.773 |

**Ressalva:**
- Os valores são **brutos**: não descontam o custo das ações de retenção, que a empresa também não definiu (*R22*). O retorno líquido (valor retido − custo da ação) entra na modelagem.
- Numa campanha de desconto, o maior custo costuma ser o desconto dado a quem **ia ficar de qualquer jeito**. A conta com esse custo está no notebook `01_eda_terra_signal.ipynb`, seção 3.5.

**O que o Ato 4 revela:** quem sai é um cliente que vale mais. A retenção deve priorizar **risco × valor**, e o ponto de corte do modelo deve ser escolhido por retorno financeiro, não por acurácia.

**➜ O gancho:** já sei quando, quem e quanto. Falta a pergunta que nenhuma coluna de contrato ou pagamento responde: **por que** eles saem? Para isso, eu preciso ouvir os próprios clientes.

---
# Ato 5 — Por que saem? (feedback classificado pela laya e revisado)

> **O motivo.** Cada cliente deixou um depoimento. Até aqui eu olhei para o que os clientes **são**; agora eu olho para o que eles **dizem**.

**Objetivo do ato:** descobrir **as categorias de saída mais frequentes** a partir do feedback dos **clientes que saíram**.

**De onde vem a classificação:** o feedback foi classificado **fora deste notebook**, pelo script `laya_feedback.py`, em três etapas:
1. **Classificação:** a [laya](https://pypi.org/project/laya/) (modelo `convaiinnovations/laya`, zero-shot, sem treino específico) leu cada um dos 5.634 feedbacks e respondeu *"What is the customer's main complaint or concern about the company?"*, escolhendo uma de 6 categorias.
2. **Revisão:** as classificações foram divididas em lotes de 200 e revisadas seguindo `revisao/INSTRUCOES.md`. Só as erradas foram registradas em `revisao/correcoes_XX.csv`.
3. **Junção:** as correções foram aplicadas, gerando o `feedback_laya_revisado.csv`, que este ato lê.

**Por quê fora do notebook:** a laya leva cerca de 1 hora no CPU e a revisão é um passo manual. Separando, este notebook roda em segundos e sempre com a mesma classificação.

**Filtro:** só os **1.504 clientes com `Churn = Yes`**. **Por quê:** a pergunta do ato é "por que saem"; o feedback de quem ficou não responde a ela.

**Ressalvas do ato:**
- Parte dos feedbacks foi escrita **depois** do cancelamento (*"I recently cancelled…"*), e a revisão viu a coluna `Churn`. Para **entender os motivos** de quem saiu, isso não é problema. Mas a classificação **não pode ser usada para prever** o churn: ela carrega a resposta (vazamento do alvo). Por isso ela não entra no modelo.
- A pesquisa foi pontual e não será repetida (*R20*). Usar isso no dia a dia exigiria uma coleta periódica.

### 5.1 · O que a base corrigida traz?

- **Como:** leio o `feedback_laya_revisado.csv` e junto com a base pelo `customerID`. As colunas que uso:
  - `atrito`: a categoria final, depois da revisão;
  - `atrito_laya`: a categoria que a laya escolheu, antes da revisão;
  - `corrigido`: se a revisão trocou a categoria.

In [0]:
# Rótulo em inglês (o que a laya vê) -> nome em português (o que aparece nas tabelas)
CATEGORIAS = {
    "service_quality": "Qualidade do serviço",
    "competitor": "Concorrência",
    "price": "Preço",
    "personal": "Motivos pessoais",
    "payment": "Pagamento",
    "satisfied": "Sem reclamação",
}

laya_df = pd.read_csv("./feedback_laya_revisado.csv", usecols=["customerID", "atrito_laya", "atrito", "corrigido"])
com_laya = df.merge(laya_df, on="customerID", how="left")
com_laya["motivo_pt"] = com_laya["atrito"].map(CATEGORIAS)
com_laya["motivo_laya_pt"] = com_laya["atrito_laya"].map(CATEGORIAS)

saiu = com_laya[com_laya["churn"] == 1].reset_index(drop=True)

pd.Series({
    "Clientes na base": len(com_laya),
    "Clientes sem classificação": com_laya["atrito"].isna().sum(),
    "Clientes que saíram": len(saiu),
    "Classificações corrigidas pela revisão (base inteira)": com_laya["corrigido"].sum(),
}, name="clientes").to_frame()

### 5.2 · O que a revisão mudou?

- **Por quê:** um classificador zero-shot erra; antes de contar os motivos, quero saber o quanto a revisão precisou corrigir.
- **Como:** cruzo a categoria da laya (linhas) com a categoria final (colunas), só para quem saiu.

In [0]:
mudancas = pd.crosstab(saiu["motivo_laya_pt"], saiu["motivo_pt"], margins=True, margins_name="Total")
print(f"Classificações corrigidas entre quem saiu: {saiu['corrigido'].sum()} de {len(saiu)} "
      f"({pct(saiu['corrigido'].mean())})")
mudancas

**O que os dados mostram:**
- A revisão corrigiu **564** das 1.504 classificações de quem saiu (37,5%). Todas as correções da base inteira foram em clientes que saíram: entre quem ficou, a laya acertou o "sem reclamação".
- As maiores trocas foram de **concorrência para qualidade do serviço** (201) e a retirada de **"sem reclamação"** de quem saiu: a laya tinha marcado 151 clientes que saíram como sem reclamação; depois da revisão, sobraram 2.

**Ressalva:** a revisão sabia que esses clientes tinham saído. Para descrever motivos de saída, isso ajuda (a pergunta já é "por que saiu?"); para prever, seria vazamento.

### 5.3 · As etiquetas fazem sentido?

- **Como:** leio um feedback de cada categoria e confiro se a etiqueta bate com o conteúdo.

In [0]:
exemplos = (saiu[saiu["motivo_pt"] != "Sem reclamação"]
            .groupby("motivo_pt").sample(1, random_state=42)[["motivo_pt", "CustomerFeedback"]])
exemplos.set_index("motivo_pt")

### 5.4 · Quais são os motivos mais comuns?

- **Como:** % das saídas da categoria = clientes que saíram com esse motivo ÷ total de clientes que saíram. Tabela e gráfico de barras, com as **3 categorias mais frequentes** em vermelho.

In [0]:
freq = saiu["motivo_pt"].value_counts().to_frame("clientes")
freq["% das saídas"] = freq["clientes"] / len(saiu)
freq["mensalidade média"] = saiu.groupby("motivo_pt")["MonthlyCharges"].mean()
display(freq.assign(**{"% das saídas": freq["% das saídas"].map(pct),
                       "mensalidade média": freq["mensalidade média"].map(brl)}))

f = freq.sort_values("clientes")
tres_maiores = set(freq.index[:3])
cores = [VERMELHO if k in tres_maiores else CINZA for k in f.index]
fig, ax = plt.subplots(figsize=(8, 3.4))
barras = ax.barh(f.index, f["% das saídas"], color=cores, height=0.6)
for b, v, n in zip(barras, f["% das saídas"], f["clientes"]):
    ax.text(v + 0.005, b.get_y() + b.get_height() / 2, f"{pct(v, 0)} (n={n})", va="center", fontsize=9)
ax.set(title=f"Por que saem: motivo de saída dos {len(saiu)} clientes que saíram",
       xlim=(0, f["% das saídas"].max() * 1.25))
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**O que os dados mostram:**
- **Qualidade do serviço (35%)**, **concorrência (32%)** e **preço (20%)** explicam **87%** das saídas.
- Motivos pessoais (8%) e pagamento (4%) pesam pouco. Quase ninguém sai "sem reclamação" (2 clientes).
- Quem sai por **preço** pagava a maior mensalidade média (R\$ 90,49); quem sai por **qualidade**, a menor (R\$ 59,32).

### 5.5 · O motivo muda com o tipo de internet?

- **Por quê:** no Ato 3, a fibra concentrava as saídas. Quero saber se quem tem fibra sai pelos mesmos motivos de quem tem DSL.
- **Como:** para cada tipo de internet, a % de cada motivo entre os clientes que saíram (cada coluna soma 100%).

In [0]:
NOMES_INTERNET = {"DSL": "DSL", "Fiber optic": "Fibra", "No": "Sem internet"}

por_internet = pd.crosstab(saiu["motivo_pt"], saiu["InternetService"].map(NOMES_INTERNET), normalize="columns")
por_internet = por_internet.loc[freq.index, ["Fibra", "DSL", "Sem internet"]]
display(saiu["InternetService"].map(NOMES_INTERNET).value_counts().rename("clientes que saíram").to_frame().T)
por_internet.map(pct)

**O que os dados mostram:**

| Quem saiu tinha… | Motivo principal | Segundo motivo |
|---|---|---|
| **Fibra** | Concorrência (37%) | Preço (28%) |
| **DSL** | **Qualidade do serviço (73%)** | Concorrência (21%) |
| **Sem internet** | Qualidade do serviço (51%) | Concorrência (24%) |

- **Fibra:** o cliente não reclama da conexão; ele acha **caro** e encontra **oferta melhor** na concorrência.
- **DSL:** o problema é a **conexão**: quase 3 de cada 4 saídas.

**O que o Ato 5 revela:** os clientes saem por **três motivos principais: qualidade do serviço, concorrência e preço**. E o motivo depende do perfil: quem tem **DSL** sai pela **qualidade da conexão**; quem tem **fibra** sai por **preço e concorrência**. Uma única ação de retenção não serve para todos.

**➜ O gancho:** com o motivo em mãos, a história fecha: dá para ligar cada sinal dos dados a uma ação.

---
# Conclusão — O desfecho: do diagnóstico às alavancas

> **A história em uma frase:** a Terra Signal perde cedo, num perfil bem definido (contrato mensal com fibra), os clientes que mais pagam, e eles saem por preço e concorrência, enquanto quem tem DSL sai pela qualidade da conexão.

### A história, ato por ato

| Ato | Pergunta | Resposta |
|---|---|---|
| 1 | Quem são os clientes? | 5.634 clientes, 26,7% saíram. Base dominada por contrato mensal (55%) e fibra (44%). |
| 2 | **Quando** saem? | Cedo: metade das saídas até o 10º mês; 49% de churn entre clientes com até 9 meses. |
| 3 | **Quem** sai? | Contrato mensal + fibra: 30% da base, 62% das saídas. |
| 4 | **Onde está o dinheiro?** | Quem sai paga 22% a mais e responde por 30,8% da soma das mensalidades. |
| 5 | **Por que** saem? | Qualidade do serviço (35%), concorrência (32%) e preço (20%). Fibra sai por preço e concorrência; DSL, pela conexão. |

### Das causas às alavancas

| Sinal dos dados | Alavanca | Público | Momento |
|---|---|---|---|
| Contrato mensal sai cerca de 13 vezes mais que o de 2 anos | Oferta de migração para contrato de 12 meses | Contrato mensal, sobretudo com fibra | Primeiros meses de relacionamento |
| Fibra sai por concorrência (37%) e preço (28%) | Oferta competitiva: desconto no plano anual, comparado ao preço da concorrência | Clientes de fibra com contrato mensal | Antes do fim de cada ciclo mensal |
| DSL sai pela qualidade da conexão (73%) | Visita técnica e oferta de migração para fibra onde houver cobertura | Clientes de DSL | Ao primeiro sinal de queixa |
| Cheque eletrônico tem 45% de churn | Incentivo ao débito ou cartão automático | Quem paga com cheque eletrônico | Na contratação e nos primeiros meses |
| 49% de churn entre clientes com até 9 meses | Acompanhamento ativo do cliente novo | Clientes com até 9 meses de casa | Do 1º ao 9º mês |
| Sem segurança online ou suporte técnico, ~42% de churn | Oferta de segurança online e suporte técnico | Clientes com internet e sem esses adicionais | Primeiros meses (testar: é associação) |
| Quem sai paga 22% a mais | Priorizar por risco × valor | Toda a lista de retenção | Sempre |

### Próximos passos

Feitos no notebook `modelo_churn.ipynb`:
1. Limpeza completa dos dados (imputação de `tenure`; `TotalCharges` vazio preenchido no pipeline).
2. Modelo de churn **sem o texto do feedback** (que tem vazamento do alvo).
3. Ponto de corte escolhido para reduzir os falsos negativos e validado pelo retorno financeiro.

Em aberto:
4. Piloto com grupo de controle para medir a taxa de sucesso real das ações de retenção.
5. Coleta periódica de feedback de clientes **ativos**, para acompanhar os motivos de insatisfação antes da saída.